In [1]:
# Hide warnings for clean output
import warnings
warnings.filterwarnings("ignore")

# Basic imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Neural network
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# UI
!pip install -q gradio
import gradio as gr

# Force CPU to avoid CUDA/GPU issues on Kaggle
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

2025-10-16 08:48:42.254971: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760604522.480457      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760604522.545463      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [4]:
# Kaggle path to dataset
data_path = "/kaggle/input/spotify-dataset-19212020-600k-tracks/tracks.csv"

# Load dataset
data = pd.read_csv(data_path)

# Select features
features = ['danceability','energy','loudness','speechiness','acousticness',
            'instrumentalness','liveness','valence','tempo']

# Create target column: hit = 1 if popularity >= 80, else 0
if 'popularity' in data.columns:
    data['billboard_hit'] = data['popularity'].apply(lambda x: 1 if x >= 80 else 0)
else:
    raise ValueError("No 'popularity' column found. Cannot create target.")

# Drop missing values in features + target
data = data.dropna(subset=features + ['billboard_hit'])

print("Dataset shape:", data.shape)
data.head()

Dataset shape: (586672, 21)


,id,name,popularity,duration_ms,explicit,artists,id_artists,release_date,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,billboard_hit
0,35iwgR4jXetI318WEWsa1Q,Carve,6,126903,0,['Uli'],['45tIt06XoI0Iio4LBEVpls'],1922-02-22,0.645,0.4450,...,-13.338,1,0.4510,0.674,0.7440,0.151,0.127,104.851,3,0
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,0,98200,0,['Fernando Pessoa'],['14jtPCOoNZwquk5wd9DxrY'],1922-06-01,0.695,0.2630,...,-22.136,1,0.9570,0.797,0.0000,0.148,0.655,102.009,1,0
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,0,181640,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.434,0.1770,...,-21.180,1,0.0512,0.994,0.0218,0.212,0.457,130.418,5,0
3,08FmqUhxtyLTn6pAh6bk45,El Prisionero - Remasterizado,0,176907,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.321,0.0946,...,-27.961,1,0.0504,0.995,0.9180,0.104,0.397,169.980,3,0
4,08y9GfoqCWfOGsKdwojr5e,Lady of the Evening,0,163080,0,['Dick Haymes'],['3BiJGZsyX9sJchTqcSA7Su'],1922,0.402,0.1580,...,-16.900,0,0.0390,0.989,0.1300,0.311,0.196,103.220,4,0


In [6]:
# Features to use
features = ['danceability', 'energy', 'key', 'loudness', 'mode',
            'speechiness', 'acousticness', 'instrumentalness', 'liveness', 
            'valence', 'tempo', 'time_signature']

# Target column
target = 'billboard_hit'  # 1 = hit, 0 = not hit

# If your dataset doesn't have 'billboard_hit', create it from popularity
# Example: songs with popularity >= 80 are hits
if 'billboard_hit' not in data.columns:
    data['billboard_hit'] = data['popularity'].apply(lambda x: 1 if x >= 80 else 0)

# Features and labels
X = data[features].values
y = data[target].values

# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training features shape:", X_train_scaled.shape)
print("Test features shape:", X_test_scaled.shape)


Training features shape: (469337, 12)
Test features shape: (117335, 12)


In [7]:
# Compute class weights for imbalanced dataset
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print("Class weights:", class_weight_dict)

Class weights: {0: 0.5008141723612493, 1: 307.56028833551767}


In [8]:
# Compute class weights for imbalance
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = {classes[i]: class_weights[i] for i in range(len(classes))}

# Build neural network
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model with class weights
history = model.fit(
    X_train_scaled, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test_scaled, y_test),
    class_weight=class_weight_dict,
    verbose=1
)

2025-10-16 08:49:40.064111: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/20
7334/7334 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.7273 - loss: 0.5511 - val_accuracy: 0.7512 - val_loss: 0.4712
Epoch 2/20
7334/7334 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.7423 - loss: 0.4765 - val_accuracy: 0.6497 - val_loss: 0.6530
Epoch 3/20
7334/7334 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.7138 - loss: 0.4798 - val_accuracy: 0.7532 - val_loss: 0.4936
Epoch 4/20
7334/7334 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.7254 - loss: 0.4759 - val_accuracy: 0.6999 - val_loss: 0.5286
Epoch 5/20
7334/7334 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - accuracy: 0.7339 - loss: 0.4573 - val_accuracy: 0.6709 - val_loss: 0.6169
Epoch 6/20
7334/7334 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.7434 - loss: 0.4439 - val_accuracy: 0.8296 - val_loss: 0.3539
Epoch 7/20
7334/7334 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.7556 - loss: 0.4569 - val_accuracy: 0.7677 - val_loss: 0.4771
Epoch 8/20
7334/7334 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - accuracy: 0.7660 - loss: 0

In [9]:
# Evaluate performance
loss, acc = model.evaluate(X_test_scaled, y_test)
print(f"Test Accuracy: {acc*100:.2f}%")

3667/3667 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.7965 - loss: 0.4496
Test Accuracy: 79.70%


In [10]:
def predict_hit(danceability, energy, loudness, speechiness, acousticness,
                instrumentalness, liveness, valence, tempo):
    
    features_input = np.array([[danceability, energy, loudness, speechiness, acousticness,
                                instrumentalness, liveness, valence, tempo]])
    
    features_scaled = scaler.transform(features_input)
    prob = model.predict(features_scaled)[0][0]
    
    return f"Predicted hit probability: {prob*100:.2f}%"

In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# Predict on test set
y_pred_prob = model.predict(X_test_scaled)          # predicted probabilities
y_pred = (y_pred_prob > 0.5).astype(int)           # convert to 0/1

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
# Precision
precision = precision_score(y_test, y_pred)
# Recall
recall = recall_score(y_test, y_pred)
# F1 Score
f1 = f1_score(y_test, y_pred)
# ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_prob)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# Print results
print("Model Evaluation Metrics:")
print("-------------------------")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall: {recall*100:.2f}%")
print(f"F1 Score: {f1*100:.2f}%")
print(f"ROC-AUC: {roc_auc*100:.2f}%")
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

3667/3667 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step 
Model Evaluation Metrics:
-------------------------
Accuracy: 79.70%
Precision: 0.63%
Recall: 78.53%
F1 Score: 1.24%
ROC-AUC: 86.23%

Confusion Matrix:
[[93367 23777]
 [   41   150]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.80      0.89    117144
           1       0.01      0.79      0.01       191

    accuracy                           0.80    117335
   macro avg       0.50      0.79      0.45    117335
weighted avg       1.00      0.80      0.89    117335



In [12]:
import gradio as gr
import pandas as pd

def predict_hit_ui(
    danceability, energy, key, loudness, mode,
    speechiness, acousticness, instrumentalness, liveness,
    valence, tempo, time_signature
):
    # Construct input dict
    sample_input = {
        'danceability': danceability,
        'energy': energy,
        'key': key,
        'loudness': loudness,
        'mode': mode,
        'speechiness': speechiness,
        'acousticness': acousticness,
        'instrumentalness': instrumentalness,
        'liveness': liveness,
        'valence': valence,
        'tempo': tempo,
        'time_signature': time_signature
    }
    
    # Convert to DataFrame and scale
    X_input = scaler.transform(pd.DataFrame([sample_input]))
    
    # Predict
    prediction = model.predict(X_input)[0][0]
    
    # Convert to percentage
    return f"{prediction*100:.2f}%"

# Create Gradio interface
ui = gr.Interface(
    fn=predict_hit_ui,
    inputs=[
        gr.Slider(0, 1, value=0.5, label="Danceability"),
        gr.Slider(0, 1, value=0.5, label="Energy"),
        gr.Number(value=5, label="Key (0-11)"),
        gr.Number(value=-5.0, label="Loudness (dB)"),
        gr.Number(value=1, label="Mode (0 or 1)"),
        gr.Slider(0, 1, value=0.05, label="Speechiness"),
        gr.Slider(0, 1, value=0.1, label="Acousticness"),
        gr.Slider(0, 1, value=0.0, label="Instrumentalness"),
        gr.Slider(0, 1, value=0.15, label="Liveness"),
        gr.Slider(0, 1, value=0.5, label="Valence"),
        gr.Number(value=120, label="Tempo"),
        gr.Number(value=4, label="Time Signature")
    ],
    outputs=gr.Textbox(label="Chance of Billboard Hit"),
    title="Billboard Hot 100 Hit Predictor",
    description="Adjust the features of a song to see the predicted probability of becoming a Billboard Hot 100 hit."
)

ui.launch()


* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://bc7c126f92de9ab1aa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
